# HER2 Affinity Prediction

This notebook evaluates sequence representations and machine-learning models for classifying high-affinity HER2-binding trastuzumab CDRH3 variants.

The analysis compares edit-distance, position-specific one-hot, and ESM-2 representations using Logistic Regression and XGBoost. Average Precision (AP) is used as the primary evaluation metric.

In [1]:
import pandas as pd

In [2]:
train_data=pd.read_csv("../data/raw/train.csv")

In [3]:
train_data.head()

,seq,label,edit_distance,class,cluster_centre
0,FFVSGFYYQA,0,7,mid,LMVSGLYYQA
1,LCVSGFYSQA,0,7,mid,LMVSGLYYQA
2,FMVSGLYYQD,0,7,mid,LMVSGLYYQA
3,LIVSGLYYKA,0,8,mid,LMVSGLYYQA
4,LMVSGFYFYA,0,7,mid,LMVSGLYYQA


In [4]:
train_data.shape

(367044, 5)

In [5]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 367044 entries, 0 to 367043
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   seq             367044 non-null  str  
 1   label           367044 non-null  int64
 2   edit_distance   367044 non-null  int64
 3   class           367044 non-null  str  
 4   cluster_centre  367044 non-null  str  
dtypes: int64(2), str(3)
memory usage: 14.0 MB


In [6]:
train_data['class'].value_counts()

class
mid     143278
high    120505
low     103261
Name: count, dtype: int64

In [7]:
train_data['label'].value_counts()

label
0    246539
1    120505
Name: count, dtype: int64

In [8]:
train_data['cluster_centre'].value_counts()

cluster_centre
WGLSGLYVYS    11623
WATLGMYVFV     3965
WSQDSIYVFA     3432
WLLSGFYVFA     3338
WTHAGMYMFS     2918
              ...  
GTARLLPTPL        1
YSLLGNGVAF        1
MELQAHTSSP        1
GSSKLNKLCI        1
LNEFGLWHVH        1
Name: count, Length: 13829, dtype: int64

In [9]:
pd.crosstab(train_data["class"], train_data["label"])


label,0,1
class,,
high,0,120505
low,103261,0
mid,143278,0


In [10]:
val_data=pd.read_csv("../data/raw/val.csv")
test_data=pd.read_csv("../data/raw/test.csv")

In [11]:
print("Train:", train_data.shape)
print("Val:", val_data.shape)
print("Test:", test_data.shape)

Train: (367044, 5)
Val: (78651, 5)
Test: (78651, 5)


In [12]:
train_data[["seq","cluster_centre"]].head(10)

,seq,cluster_centre
0,FFVSGFYYQA,LMVSGLYYQA
1,LCVSGFYSQA,LMVSGLYYQA
2,FMVSGLYYQD,LMVSGLYYQA
3,LIVSGLYYKA,LMVSGLYYQA
4,LMVSGFYFYA,LMVSGLYYQA
5,LIVSGLYYKD,LMVSGLYYQA
6,LLVSGSYYYA,LMVSGLYYQA
7,LIVSGLYYQA,LMVSGLYYQA
8,LMVSGLYYQA,LMVSGLYYQA
9,LMVSGLYFQA,LMVSGLYYQA


In [13]:
train_data["cluster_centre"].nunique()


13829

In [14]:
test_data["cluster_centre"].nunique()


706

In [15]:
val_data["cluster_centre"].nunique()

251

In [16]:
train_clusters = set(train_data["cluster_centre"])
val_clusters = set(val_data["cluster_centre"])
test_clusters = set(test_data["cluster_centre"])

In [17]:
print("Train-Val overlap:", len(train_clusters & val_clusters))
print("Train-Test overlap:", len(train_clusters & test_clusters))
print("Val-Test overlap:", len(val_clusters & test_clusters))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [18]:
train_data["seq"].str.len().value_counts().sort_index()

seq
10    367044
Name: count, dtype: int64

In [19]:
print("Train duplicates:", train_data["seq"].duplicated().sum())
print("Val duplicates:", val_data["seq"].duplicated().sum())
print("Test duplicates:", test_data["seq"].duplicated().sum())

Train duplicates: 0
Val duplicates: 0
Test duplicates: 0


In [20]:
train_seqs = set(train_data["seq"])
val_seqs = set(val_data["seq"])
test_seqs = set(test_data["seq"])

print("Train-Val sequence overlap:", len(train_seqs & val_seqs))
print("Train-Test sequence overlap:", len(train_seqs & test_seqs))
print("Val-Test sequence overlap:", len(val_seqs & test_seqs))


Train-Val sequence overlap: 0
Train-Test sequence overlap: 0
Val-Test sequence overlap: 0


In [21]:
print("Train:\n", train_data["class"].value_counts(normalize=True))
print("Validation:\n", val_data["class"].value_counts(normalize=True))
print("Test:\n", test_data["class"].value_counts(normalize=True))

Train:
 class
mid     0.390356
high    0.328312
low     0.281331
Name: proportion, dtype: float64
Validation:
 class
low     0.434832
high    0.328311
mid     0.236856
Name: proportion, dtype: float64
Test:
 class
mid     0.357325
high    0.328311
low     0.314363
Name: proportion, dtype: float64


In [22]:
print("Train:\n", train_data["label"].value_counts(normalize=True))
print("Validation:\n", val_data["label"].value_counts(normalize=True))
print("Test:\n", test_data["label"].value_counts(normalize=True))

Train:
 label
0    0.671688
1    0.328312
Name: proportion, dtype: float64
Validation:
 label
0    0.671689
1    0.328311
Name: proportion, dtype: float64
Test:
 label
0    0.671689
1    0.328311
Name: proportion, dtype: float64


In [23]:
print("Train:\n", train_data.isna().sum())
print("\nValidation:\n", val_data.isna().sum())
print("\nTest:\n", test_data.isna().sum())


Train:
 seq               0
label             0
edit_distance     0
class             0
cluster_centre    0
dtype: int64

Validation:
 seq               0
label             0
edit_distance     0
class             0
cluster_centre    0
dtype: int64

Test:
 seq               0
label             0
edit_distance     0
class             0
cluster_centre    0
dtype: int64


In [24]:
print(
    "Train:\n",
    train_data["edit_distance"].value_counts(normalize=True).sort_index()
)

print(
    "Validation:\n",
    val_data["edit_distance"].value_counts(normalize=True).sort_index()
)

print(
    "Test:\n",
    test_data["edit_distance"].value_counts(normalize=True).sort_index()
)

Train:
 edit_distance
2     0.000616
3     0.001624
4     0.008124
5     0.047019
6     0.161812
7     0.314919
8     0.292578
9     0.134586
10    0.038723
Name: proportion, dtype: float64
Validation:
 edit_distance
1     0.000636
2     0.004361
3     0.009536
4     0.026306
5     0.049828
6     0.139769
7     0.284637
8     0.320072
9     0.150882
10    0.013973
Name: proportion, dtype: float64
Test:
 edit_distance
2     0.000394
3     0.001831
4     0.017050
5     0.093235
6     0.204104
7     0.281649
8     0.232356
9     0.113095
10    0.056287
Name: proportion, dtype: float64


In [25]:
from pathlib import Path
from her2clf.data import load_splits

In [26]:
data_dir=Path("../data/raw")
train_data, val_data, test_data=load_splits(data_dir)

In [27]:
y_train = train_data["label"]
y_val = val_data["label"]
y_test = test_data["label"]

### Baseline: One-hot + Logistic Regression

In [28]:
from her2clf.features import fit_onehot_encoder, transform_sequences

X_train, encoder=fit_onehot_encoder(train_data["seq"])
X_val= transform_sequences(val_data["seq"], encoder)
X_test= transform_sequences(test_data["seq"], encoder)

In [29]:
from her2clf.models import train_logistic_regression

logistic_model = train_logistic_regression(X_train, y_train)
val_prob=logistic_model.predict_proba(X_val)[:,1]

In [30]:
from sklearn.metrics import roc_auc_score, average_precision_score
print("ROC-AUC:", roc_auc_score(y_val, val_prob))
print("PR-AUC:", average_precision_score(y_val, val_prob))

ROC-AUC: 0.9643616850123388
PR-AUC: 0.9364805889406737


Representation: position-specific one-hot encoding (200 features)
Model: Logistic Regression
Split: clonotype split

Validation:
- ROC-AUC: 0.9644
- PR-AUC: 0.9365

### One-hot Logistic Regression: Performance by Edit Distance

In [31]:
val_results=val_data[["seq", "label", "edit_distance"]].copy()
val_results["prob"]=val_prob
val_results.head()

,seq,label,edit_distance,prob
0,WYALPFYKYD,1,6,0.853679
1,WYAQPFYYYD,1,6,0.830298
2,WYAQAFYYYD,1,6,0.734318
3,WNAQAFYIYD,1,6,0.366453
4,GALLAATLRL,0,10,0.521642


In [32]:
val_results["edit_distance"].value_counts().sort_index()

edit_distance
1        50
2       343
3       750
4      2069
5      3919
6     10993
7     22387
8     25174
9     11867
10     1099
Name: count, dtype: int64

In [33]:
from sklearn.metrics import roc_auc_score, average_precision_score

results_by_distance = []

for distance, group in val_results.groupby("edit_distance"):
    roc_auc = roc_auc_score(group["label"], group["prob"])
    pr_auc = average_precision_score(group["label"], group["prob"])

    results_by_distance.append({
        "edit_distance": distance,
        "n": len(group),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    })

results_by_distance = pd.DataFrame(results_by_distance)
results_by_distance

,edit_distance,n,roc_auc,pr_auc
0,1,50,0.593750,0.979641
1,2,343,0.602683,0.981791
2,3,750,0.720351,0.961597
3,4,2069,0.740550,0.924762
4,5,3919,0.787574,0.871121
5,6,10993,0.872994,0.858768
6,7,22387,0.954570,0.940434
7,8,25174,0.989073,0.978015
8,9,11867,0.988927,0.963728
9,10,1099,0.472659,0.093916


In [34]:
val_results.groupby("edit_distance")["label"].agg(
    n="count",
    positives="sum",
    positive_rate="mean"
)

,n,positives,positive_rate
edit_distance,,,
1,50,48,0.960000
2,343,332,0.967930
3,750,683,0.910667
4,2069,1705,0.824070
5,3919,2468,0.629752
6,10993,4573,0.415992
7,22387,8376,0.374146
8,25174,6686,0.265591
9,11867,938,0.079043


### Performance by edit distance

High-affinity prevalence decreases strongly with distance from parental trastuzumab
(~96% at distance 1 to ~1.2% at distance 10).

Because positive prevalence varies substantially across edit-distance groups, PR-AUC
should be interpreted relative to each group's positive rate. The one-hot logistic
regression performs particularly strongly at distances 7–9, while performance at
distance 10 is poor/uncertain and based on only 13 positive examples.

Edit distance is not provided to the model, but is inherently encoded by the sequence.
A later baseline using edit distance alone will test how much predictive signal can be
explained simply by distance from the parent sequence.

In [35]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, val_prob)

In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (val_prob >= threshold).astype(int)

    print(
        threshold,
        "Precision:", round(precision_score(y_val, pred), 3),
        "Recall:", round(recall_score(y_val, pred), 3),
        "F1:", round(f1_score(y_val, pred), 3)
    )

0.2 Precision: 0.603 Recall: 0.983 F1: 0.748
0.3 Precision: 0.689 Recall: 0.963 F1: 0.803
0.4 Precision: 0.76 Recall: 0.935 F1: 0.838
0.5 Precision: 0.82 Recall: 0.895 F1: 0.856
0.6 Precision: 0.872 Recall: 0.842 F1: 0.857
0.7 Precision: 0.916 Recall: 0.768 F1: 0.836
0.8 Precision: 0.953 Recall: 0.652 F1: 0.774


### Edit-Distance-Only Baseline

In [37]:
X_train_dist = train_data[["edit_distance"]]
X_val_dist = val_data[["edit_distance"]]

In [38]:
from sklearn.linear_model import LogisticRegression

distance_model = LogisticRegression()
distance_model.fit(X_train_dist, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [39]:
val_prob_dist = distance_model.predict_proba(X_val_dist)[:, 1]

In [40]:
print("ROC-AUC:", roc_auc_score(y_val, val_prob_dist))
print("PR-AUC:", average_precision_score(y_val, val_prob_dist))

ROC-AUC: 0.6992978442308722
PR-AUC: 0.5176465039300809



To test whether sequence-model performance is primarily explained by distance
from parental trastuzumab, logistic regression was trained using edit distance
as the only feature.

Validation:
- ROC-AUC: 0.6993
- PR-AUC: 0.5176

Edit distance contains substantial predictive signal, but performs far below
the position-specific one-hot sequence baseline (ROC-AUC 0.9644, PR-AUC 0.9365).
Therefore, the sequence model captures information beyond mutation count alone.

### XGBoost baseline

In [41]:
from her2clf.models import train_xgboost

In [42]:
xgb_model = train_xgboost(X_train, y_train)
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val, xgb_val_prob))
print("PR-AUC:", average_precision_score(y_val, xgb_val_prob))

ROC-AUC: 0.991303058541407
PR-AUC: 0.9844452164483588


Representation: position-specific one-hot encoding (200 features)  
Model: XGBoost classifier  
Split: clonotype split

Validation:
- ROC-AUC: 0.9913
- PR-AUC: 0.9844

XGBoost substantially outperforms logistic regression using the same
one-hot sequence representation, suggesting that nonlinear interactions
between sequence positions contain important predictive information.

## ESM-2 Representations

In [43]:
from transformers import AutoTokenizer

/Users/Elnaz/Documents/ml_projects/her2-affinity-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
model_name="facebook/esm2_t6_8M_UR50D"
tokenizer=AutoTokenizer.from_pretrained(model_name)

### Exploring ESM-2 Token Embeddings

In [45]:
example_sequences=train_data["seq"].head(5).tolist()

In [46]:
example_sequences

['FFVSGFYYQA', 'LCVSGFYSQA', 'FMVSGLYYQD', 'LIVSGLYYKA', 'LMVSGFYFYA']

In [47]:
tokens= tokenizer(
    example_sequences,
    padding=True,
    return_tensors="pt"
)

tokens

{'input_ids': tensor([[ 0, 18, 18,  7,  8,  6, 18, 19, 19, 16,  5,  2],
        [ 0,  4, 23,  7,  8,  6, 18, 19,  8, 16,  5,  2],
        [ 0, 18, 20,  7,  8,  6,  4, 19, 19, 16, 13,  2],
        [ 0,  4, 12,  7,  8,  6,  4, 19, 19, 15,  5,  2],
        [ 0,  4, 20,  7,  8,  6, 18, 19, 18, 19,  5,  2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [48]:
tokens["input_ids"].shape

torch.Size([5, 12])

In [49]:
import torch
from transformers import AutoModel

In [50]:
model=AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D")
model.eval()

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8340.20it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
          (dens

In [51]:
with torch.no_grad():
    outputs=model(**tokens)

In [52]:
outputs.last_hidden_state.shape

torch.Size([5, 12, 320])

In [53]:
amino_acid_embeddings=outputs.last_hidden_state[:,1:-1,:]

In [54]:
sequence_embeddings=amino_acid_embeddings.mean(dim=1)
sequence_embeddings.shape

torch.Size([5, 320])

In [55]:
print("MPS available:", torch.backends.mps.is_available())

MPS available: False


In [56]:
import time

sample_sequences = train_data["seq"].head(1000).tolist()

sample_tokens = tokenizer(
    sample_sequences,
    padding=True,
    return_tensors="pt",
)

start = time.time()

with torch.no_grad():
    sample_outputs = model(**sample_tokens)

sample_embeddings = sample_outputs.last_hidden_state[:, 1:-1, :].mean(dim=1)

elapsed = time.time() - start

print("Shape:", sample_embeddings.shape)
print("Time:", elapsed)

Shape: torch.Size([1000, 320])
Time: 0.6410219669342041


### Mean-Pooled ESM-2 Embeddings

In [57]:
from her2clf.esm import load_esm_model, embed_sequences

In [58]:
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer, esm_model = load_esm_model(model_name)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 12148.77it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [59]:
sample_sequences= train_data["seq"].head(1000).tolist()
sample_embedding=embed_sequences(sample_sequences,tokenizer,esm_model)

In [60]:
type(sample_embedding)

numpy.ndarray

In [61]:
sample_embedding.shape

(1000, 320)

In [62]:
train_sequences = train_data["seq"].tolist()
val_sequences = val_data["seq"].tolist()

In [63]:
X_train_esm = embed_sequences(
    train_sequences,
    tokenizer,
    esm_model,
)

X_val_esm = embed_sequences(
    val_sequences,
    tokenizer,
    esm_model,
)

In [64]:
print(X_train_esm.shape)
print(X_val_esm.shape)

(367044, 320)
(78651, 320)


In [65]:
esm_logistic_model = train_logistic_regression(X_train_esm,y_train)

In [66]:
esm_val_prob = esm_logistic_model.predict_proba(X_val_esm)[:, 1]

In [67]:
print("ROC-AUC:", roc_auc_score(y_val, esm_val_prob))
print("PR-AUC:", average_precision_score(y_val, esm_val_prob))

ROC-AUC: 0.949471774094757
PR-AUC: 0.9046102049094623


In [68]:
xgb_model = train_xgboost(X_train_esm, y_train)
xgb_val_esm_prob = xgb_model.predict_proba(X_val_esm)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val, xgb_val_esm_prob))
print("PR-AUC:", average_precision_score(y_val, xgb_val_esm_prob))

ROC-AUC: 0.9639808289237966
PR-AUC: 0.9329446843457736


Frozen ESM-2 comparison

I compared frozen ESM-2 (8M) embeddings with the one-hot sequence representation used for the baseline models.

Representation	Classifier	ROC-AUC	PR-AUC
One-hot	Logistic Regression	0.9644	0.9365
ESM-2 8M	Logistic Regression	0.9495	0.9046
One-hot	XGBoost	0.9913	0.9844
ESM-2 8M	XGBoost	0.9640	0.9329

The ESM embeddings did not improve performance in this case. The best result came from XGBoost using the one-hot representation.

One possible reason is that the sequences are only 10 amino acids long and the training set is very large. The one-hot representation keeps the exact amino acid at each position, which seems to be very informative for this dataset. Mean pooling the ESM embeddings may also remove some useful position-specific information.

This only applies to the small frozen ESM-2 model and mean-pooling approach tested here. Other ESM models, pooling methods, or fine-tuning could give different results.

In [69]:
val_results_xgb = val_data[
    ["seq", "label", "class", "edit_distance", "cluster_centre"]
].copy()

val_results_xgb["prob"] = xgb_val_prob

In [70]:
xgb_by_distance = []

for distance, group in val_results_xgb.groupby("edit_distance"):
    xgb_by_distance.append({
        "edit_distance": distance,
        "n": len(group),
        "positive_rate": group["label"].mean(),
        "roc_auc": roc_auc_score(group["label"], group["prob"]),
        "pr_auc": average_precision_score(group["label"], group["prob"]),
    })

pd.DataFrame(xgb_by_distance)

,edit_distance,n,positive_rate,roc_auc,pr_auc
0,1,50,0.960000,0.578125,0.963305
1,2,343,0.967930,0.837623,0.991844
2,3,750,0.910667,0.879002,0.983123
3,4,2069,0.824070,0.854815,0.959865
4,5,3919,0.629752,0.920511,0.951008
5,6,10993,0.415992,0.968721,0.962323
6,7,22387,0.374146,0.990590,0.987285
7,8,25174,0.265591,0.997824,0.995713
8,9,11867,0.079043,0.995447,0.980888
9,10,1099,0.011829,0.878028,0.096139


### Position-Preserving (Flattened) ESM-2 Embeddings

In [71]:
sample_embeddings_flat = embed_sequences(
    train_data["seq"].head(1000).tolist(),
    tokenizer,
    esm_model,
    pooling="flatten",
)

print(sample_embeddings_flat.shape)

(1000, 3200)


Flattened ESM-2 embeddings preserve residue-level positional information, increasing each sequence representation from 320 to 3,200 features. Because generating the full training representation in memory was impractical, the flattened embeddings were generated and stored as disk-backed NumPy arrays.

In [72]:
del sample_embeddings_flat

In [73]:
from her2clf.esm import embed_sequences_to_disk

In [74]:
sample_flat_disk = embed_sequences_to_disk(
    train_data["seq"].head(1000).tolist(),
    tokenizer,
    esm_model,
    output_path="../data/sample_esm_flat.dat",
    pooling="flatten",
)

print(sample_flat_disk.shape)
print(type(sample_flat_disk))

(1000, 3200)
<class 'numpy.memmap'>


In [75]:
del sample_flat_disk

In [76]:
import os
os.remove("../data/sample_esm_flat.dat")

In [77]:
X_train_esm_flat = embed_sequences_to_disk(
    train_data["seq"].tolist(),
    tokenizer,
    esm_model,
    output_path="../data/train_esm_flat.dat",
    pooling="flatten",
)

In [78]:
print(X_train_esm_flat.shape)
print(type(X_train_esm_flat))

(367044, 3200)
<class 'numpy.memmap'>


In [79]:
X_val_esm_flat = embed_sequences_to_disk(
    val_data["seq"].tolist(),
    tokenizer,
    esm_model,
    output_path="../data/val_esm_flat.dat",
    pooling="flatten",
)

print(X_val_esm_flat.shape, type(X_val_esm_flat))

(78651, 3200) <class 'numpy.memmap'>


In [81]:
from her2clf.esm import load_embeddings_from_disk

X_train_esm_flat = load_embeddings_from_disk(
    "../data/train_esm_flat.dat",
    n_sequences=len(train_data),
    embedding_dim=3200,
)

X_val_esm_flat = load_embeddings_from_disk(
    "../data/val_esm_flat.dat",
    n_sequences=len(val_data),
    embedding_dim=3200,
)

print(X_train_esm_flat.shape)
print(X_val_esm_flat.shape)

(367044, 3200)
(78651, 3200)


In [82]:
esm_flat_lr = train_logistic_regression(
    X_train_esm_flat,
    y_train,
)

KeyboardInterrupt: 

In [ ]:
val_prob_esm_flat = esm_flat_lr.predict_proba(
    X_val_esm_flat
)[:, 1]

roc_auc_esm_flat = roc_auc_score(
    y_val,
    val_prob_esm_flat,
)

pr_auc_esm_flat = average_precision_score(
    y_val,
    val_prob_esm_flat,
)

print("ROC-AUC:", roc_auc_esm_flat)
print("PR-AUC:", pr_auc_esm_flat)

ESM representation comparison — Logistic Regression

Representation

| One-hot (position-specific) | 0.9644 | 0.9365 |
| ESM mean-pooled | 0.9495 | 0.9046 |
| ESM flattened (position-specific) | 0.9857 | 0.9736 |

Observation: Preserving residue position substantially improves ESM performance. Mean pooling likely removes position-specific information important for CDRH3 binding prediction. With positional information preserved, ESM also outperforms position-specific one-hot features.

Compute note: The flattened ESM experiment was run on a Google Cloud Compute Engine VM (8 vCPU, 32 GB RAM) rather than locally because the full training representation has shape (367,044, 3,200) and exceeds the practical memory capacity of the local machine. The precomputed embeddings were stored and loaded as disk-backed NumPy memmaps.

In [ ]:
esm_flat_xgb = train_xgboost(
    X_train_esm_flat,
    y_train,
)

In [ ]:
val_prob_esm_flat_xgb = esm_flat_xgb.predict_proba(
    X_val_esm_flat
)[:, 1]

roc_auc_esm_flat_xgb = roc_auc_score(
    y_val,
    val_prob_esm_flat_xgb,
)

pr_auc_esm_flat_xgb = average_precision_score(
    y_val,
    val_prob_esm_flat_xgb,
)

print("ROC-AUC:", roc_auc_esm_flat_xgb)
print("PR-AUC:", pr_auc_esm_flat_xgb)

ESM representation comparison — XGBoost

| Representation | ROC-AUC | PR-AUC |
|---|---:|---:|
| One-hot (position-specific) | 0.99130 | 0.98445 |
| ESM mean-pooled | 0.96398 | 0.93294 |
| ESM flattened (position-specific) | 0.99154 | 0.98458 |

**Observation:** Preserving residue position substantially improves ESM performance compared with mean pooling. However, unlike Logistic Regression, flattened ESM provides almost no improvement over position-specific one-hot features with XGBoost.

This suggests that XGBoost can already capture most of the useful nonlinear sequence signal from position-specific one-hot features, while the richer pretrained ESM representation provides little additional discriminative benefit for this model.

**Compute note:** The flattened ESM experiment was run on a Google Cloud Compute Engine VM (8 vCPU, 32 GB RAM) using disk-backed NumPy memmaps for the `(367,044, 3,200)` training representation.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from her2clf.data import load_splits
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

_, val_data, _ = load_splits(Path("data/raw"))
y_val = val_data["label"].values

val_prob_esm_flat_xgb = np.load(
    "artifacts/val_prob_esm_flat_xgb.npy"
)

val_prob_esm_flat_lr = np.load(
    "artifacts/val_prob_esm_flat_lr.npy"
)

### Precision–Recall Curve

![Precision–Recall Curve](../figures/esm_flat_pr_curve.png)

### ROC Curve

![ROC Curve](../figures/esm_flat_roc_curve.png)

Validation-set evaluation

The two strongest flattened-ESM models were evaluated using ROC and precision–recall curves.

| Model | ROC-AUC | PR-AUC |
|---|---:|---:|
| Logistic Regression + ESM flat | 0.9857 | 0.9736 |
| XGBoost + ESM flat | **0.9915** | **0.9846** |

XGBoost outperformed Logistic Regression across both ROC and precision–recall analysis. The difference was particularly visible at high recall, where XGBoost maintained higher precision. XGBoost was therefore selected as the primary model for further validation analysis.



### Decision-threshold analysis

In [109]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    y_pred = (val_prob_esm_flat_xgb >= threshold).astype(int)

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

     print(threshold, precision, recall, f1)

NameError: name 'val_prob_esm_flat_xgb' is not defined

Changing the classification threshold produced the expected precision–recall tradeoff. Lower thresholds recovered more true high binders but introduced more false positives, while higher thresholds increased precision at the cost of recall.

Among the thresholds evaluated, 0.6 produced the highest F1 score:

- Precision: 0.9423
- Recall: 0.9341
- F1: 0.9382

However, the threshold was not selected solely by maximizing F1. The appropriate operating point depends on the intended screening application and the relative cost of false positives versus false negatives.

### Probability calibration

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

prob_true, prob_pred = calibration_curve(
    y_val,
    val_prob_esm_flat_xgb,
    n_bins=10,
    strategy="uniform"
)

brier = brier_score_loss(
    y_val,
    val_prob_esm_flat_xgb
)

print("Brier score:", brier)
print("Predicted:", prob_pred)
print("Observed:", prob_true)

![XGBoost calibration curve](../figures/esm_flat_xgb_calibration.png)

XGBoost probabilities were also evaluated for calibration.

- Brier score: **0.0338**
- Predicted probabilities were generally higher than the observed positive frequency in the intermediate probability range.
- Calibration was considerably closer at the high-confidence end of the prediction range.

Thus, although the model shows excellent discrimination and ranking performance, its raw scores should not automatically be interpreted as perfectly calibrated probabilities.

## Error analysis

To better understand where the selected XGBoost model fails, validation errors were examined at a decision threshold of 0.6. The analysis focused on false positives and false negatives, edit distance from the cluster centre, and the original high/mid/low affinity classes.

In [ ]:
threshold = 0.6

y_pred = (val_prob_esm_flat_xgb >= threshold).astype(int)

val_analysis = val_data.copy()
val_analysis["prob_high"] = val_prob_esm_flat_xgb
val_analysis["predicted_label"] = y_pred
val_analysis["correct"] = (
    val_analysis["label"] == val_analysis["predicted_label"]
)

false_positives = val_analysis[
    (val_analysis["label"] == 0) &
    (val_analysis["predicted_label"] == 1)
]

false_negatives = val_analysis[
    (val_analysis["label"] == 1) &
    (val_analysis["predicted_label"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

In [ ]:
false_positives.sort_values(
    "prob_high",
    ascending=False
)[
    ["seq", "class", "edit_distance", "cluster_centre", "prob_high"]
].head(10)

In [ ]:
false_negatives.sort_values(
    "prob_high",
    ascending=True
)[
    ["seq", "class", "edit_distance", "cluster_centre", "prob_high"]
].head(10)

### Most confident errors

The most confident false positives and false negatives were inspected to identify sequences for which the model failed with high confidence. These examples are useful candidates for further sequence-level investigation, but high-confidence errors alone do not establish label noise or a specific biological failure mechanism.

### Error analysis by original affinity class

Although the model was trained as a binary classifier (`high` vs. `mid + low`), errors and predicted scores were also examined separately for the original high-, mid-, and low-affinity classes.

In [ ]:
class_summary = (
    val_analysis
    .groupby("class")
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_prob_high=("prob_high", "mean")
    )
)

class_summary

       n      accuracy    mean_prob_high
high   25822   0.934126     0.906606
low    34200   0.982281     0.065917
mid    18629   0.953299     0.123766

At a decision threshold of 0.6:

- High-affinity sequences had a **6.59% false-negative rate**.
- Mid-affinity sequences had a **4.67% false-positive rate**.
- Low-affinity sequences had a **1.77% false-positive rate**.

Mid-affinity sequences were therefore more likely than low-affinity sequences to be incorrectly classified as high.

Interestingly, although the model was trained only to distinguish `high` from `mid + low`, its mean predicted high-binding score retained the original affinity ordering:

**low (0.066) < mid (0.124) << high (0.907)**

This is consistent with the model retaining some affinity-related information within the negative class. However, differences in mean scores alone are not sufficient to interpret the model output as a continuous measure of affinity.

## XGBoost Hyperparameter Tuning

After selecting position-specific one-hot encoding with XGBoost as the final model family, hyperparameters were tuned on the training set using 5-fold Stratified Group K-Fold cross-validation, with Average Precision (AP) as the selection metric. `cluster_centre` was used as the grouping variable to keep related sequences within the same fold.

In [83]:
print(train_data.columns.tolist())
print(train_data.shape)

train_data.head()

['seq', 'label', 'edit_distance', 'class', 'cluster_centre']
(367044, 5)


,seq,label,edit_distance,class,cluster_centre
0,FFVSGFYYQA,0,7,mid,LMVSGLYYQA
1,LCVSGFYSQA,0,7,mid,LMVSGLYYQA
2,FMVSGLYYQD,0,7,mid,LMVSGLYYQA
3,LIVSGLYYKA,0,8,mid,LMVSGLYYQA
4,LMVSGFYFYA,0,7,mid,LMVSGLYYQA


In [84]:
groups = train_data["cluster_centre"]

print("Number of sequences:", len(train_data))
print("Number of clusters:", groups.nunique())

print("\nSequences per cluster:")
print(groups.value_counts().describe())

Number of sequences: 367044
Number of clusters: 13829

Sequences per cluster:
count    13829.000000
mean        26.541615
std        161.308362
min          1.000000
25%          1.000000
50%          2.000000
75%          9.000000
max      11623.000000
Name: count, dtype: float64


In [85]:
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from xgboost import XGBClassifier

cv= StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


In [86]:
param_distributions= {
    "n_estimators": [100, 300, 500], 
    "max_depth":  [3, 6, 9], 
    "learning_rate": [0.03, 0.1, 0.3], 
    "subsample": [0.8, 1], 
    "colsample_bytree": [0.8, 1]
}

xgb_tune= XGBClassifier(
    random_state=42, 
    n_jobs=-1
)

In [87]:
random_search=RandomizedSearchCV(
    estimator=xgb_tune,
    param_distributions=param_distributions,
    n_iter=15, 
    scoring="average_precision",
    cv=cv, 
    random_state=42, 
    n_jobs=-1, 
    verbose=2,
    refit=True)

In [88]:
random_search.fit(X_train, y_train, groups= groups)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END colsample_bytree=1, learning_rate=0.1, max_depth=3, n_estimators=500, subsample=1; total time=   8.1s
[CV] END colsample_bytree=1, learning_rate=0.1, max_depth=3, n_estimators=500, subsample=1; total time=   8.2s
[CV] END colsample_bytree=1, learning_rate=0.1, max_depth=3, n_estimators=500, subsample=1; total time=   8.3s
[CV] END colsample_bytree=1, learning_rate=0.1, max_depth=3, n_estimators=500, subsample=1; total time=   8.3s
[CV] END colsample_bytree=1, learning_rate=0.1, max_depth=3, n_estimators=500, subsample=1; total time=   8.4s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=6, n_estimators=500, subsample=0.8; total time=  14.8s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=6, n_estimators=500, subsample=0.8; total time=  15.0s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=6, n_estimators=500, subsample=0.8; total time=  15.3s
[CV] END colsample_bytree=0.8, learn

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1], 'learning_rate': [0.03, 0.1, ...], 'max_depth': [3, 6, ...], 'n_estimators': [100, 300, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedGro... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``re

In [89]:
random_search.best_score_

np.float64(0.9509891048663406)

In [90]:
random_search.best_params_


{'subsample': 1,
 'n_estimators': 300,
 'max_depth': 6,
 'learning_rate': 0.3,
 'colsample_bytree': 0.8}

In [91]:
tuned_model=random_search.best_estimator_
tuned_val_prob= tuned_model.predict_proba(X_val)[:,1]
tuned_val_pr_auc = average_precision_score(
    y_val,
    tuned_val_prob
)

tuned_val_roc_auc = roc_auc_score(
    y_val,
    tuned_val_prob
)

print("Tuned validation PR-AUC:", tuned_val_pr_auc)
print("Tuned validation ROC-AUC:", tuned_val_roc_auc)

Tuned validation PR-AUC: 0.9917959003488415
Tuned validation ROC-AUC: 0.9955739210780503


### Hyperparameter tuning result

Hyperparameters were tuned on the training set using 5-fold Stratified Group K-Fold cross-validation, with `cluster_centre` used to keep related sequences within the same fold. Randomized search optimized average precision (AP).

The selected XGBoost configuration was:
- `n_estimators = 300`
- `max_depth = 6`
- `learning_rate = 0.3`
- `subsample = 1.0`
- `colsample_bytree = 0.8`

The tuned model achieved:
- Validation AP: **0.9918**
- Validation ROC-AUC: **0.9956**

This improved over the default one-hot XGBoost model (validation AP ≈ 0.984), supporting the use of the tuned configuration for the final model.

In [92]:
import numpy as np
from scipy.sparse import vstack
X_trainval=vstack([X_train, X_val])
y_trainval=np.concatenate([y_train,y_val], axis=0)

print(X_train.shape)
print(X_val.shape)
print(X_trainval.shape)
print(X_test.shape)

(367044, 200)
(78651, 200)
(445695, 200)
(78651, 200)


In [93]:
final_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.3,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

final_model.fit(X_trainval,y_trainval)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [94]:
final_test_prob=final_model.predict_proba(X_test)[:,1]

test_roc_auc = roc_auc_score(y_test, final_test_prob)
test_pr_auc = average_precision_score(y_test, final_test_prob)

print("Final Test ROC-AUC:", test_roc_auc)
print("Final Test PR-AUC:", test_pr_auc)

Final Test ROC-AUC: 0.9912554860756493
Final Test PR-AUC: 0.9836040289357938


### Final test performance

After hyperparameter selection, the final XGBoost model was retrained on the combined training and validation sets and evaluated once on the held-out test set.

- **Test Average Precision (AP): 0.9836**
- **Test ROC-AUC: 0.9913**

The final model retained strong performance on the held-out test set, supporting good generalization beyond the data used for model and hyperparameter selection.

In [95]:
results = pd.DataFrame({
    "Model": [
        "Edit-distance Logistic Regression",
        "One-hot Logistic Regression",
        "ESM mean-pooled Logistic Regression",
        "ESM flattened Logistic Regression",
        "ESM mean-pooled XGBoost",
        "One-hot XGBoost",
        "ESM flattened XGBoost",
        "Tuned one-hot XGBoost",
    ],
    "ROC-AUC": [
        0.699,
        0.9644,
        0.9495,
        0.9857,
        0.96398,
        0.99130,
        0.99154,
        0.99126,
    ],
    "Average Precision": [
        0.518,
        0.9365,
        0.9046,
        0.9736,
        0.93294,
        0.98445,
        0.98458,
        0.98360,
    ],
    "Evaluation": [
        "Validation",
        "Validation",
        "Validation",
        "Validation",
        "Validation",
        "Validation",
        "Validation",
        "Final test",
    ],
})

results

,Model,ROC-AUC,Average Precision,Evaluation
0,Edit-distance Logistic Regression,0.69900,0.51800,Validation
1,One-hot Logistic Regression,0.96440,0.93650,Validation
2,ESM mean-pooled Logistic Regression,0.94950,0.90460,Validation
3,ESM flattened Logistic Regression,0.98570,0.97360,Validation
4,ESM mean-pooled XGBoost,0.96398,0.93294,Validation
5,One-hot XGBoost,0.99130,0.98445,Validation
6,ESM flattened XGBoost,0.99154,0.98458,Validation
7,Tuned one-hot XGBoost,0.99126,0.98360,Final test


ESM embeddings were beneficial when paired with a linear classifier, provided positional information was preserved. However, with XGBoost, position-specific one-hot encoding matched flattened ESM performance almost exactly, suggesting that the nonlinear learner could extract most of the discriminative signal needed for this task directly from residue identity and position.

In [97]:
import joblib

artifacts_dir=Path("../artifacts")
artifacts_dir.mkdir(exist_ok=True)

final_model.save_model(artifacts_dir / "xgboost_model.json")
joblib.dump(encoder, artifacts_dir / "onehot_encoder.joblib")

print("Saved model and encoder to:", artifacts_dir)

Saved model and encoder to: ../artifacts
